In [ ]:
import os, random

BASE = "/kaggle/working/data"

clean_dir = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/CLEAN"
noisy_dir = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TRAIN/NOISE"
test_dir  = "/kaggle/input/datasets/khabiphmhng/noise-speech/NOISE SPEECH/TEST "

clean_files = set(os.listdir(clean_dir))
noisy_files = set(os.listdir(noisy_dir))
common = sorted(clean_files & noisy_files)
print(f"So cap file khop ten giua CLEAN/NOISE: {len(common)}")
if len(common) == 0:
    raise RuntimeError("Khong co file nao trung ten giua CLEAN va NOISE - kiem tra lai ten file.")

random.seed(42)
random.shuffle(common)
n_valid = max(1, int(0.1 * len(common)))
valid_files = common[:n_valid]
train_files = common[n_valid:]

for split, files in [("train", train_files), ("valid", valid_files)]:
    for sub in ["clean", "noisy"]:
        os.makedirs(f"{BASE}/{split}/{sub}", exist_ok=True)
    for f in files:
        dst_clean = f"{BASE}/{split}/clean/{f}"
        dst_noisy = f"{BASE}/{split}/noisy/{f}"
        if not os.path.exists(dst_clean):
            os.symlink(os.path.join(clean_dir, f), dst_clean)
        if not os.path.exists(dst_noisy):
            os.symlink(os.path.join(noisy_dir, f), dst_noisy)

os.makedirs(f"{BASE}/test_infer/noisy", exist_ok=True)
for f in os.listdir(test_dir):
    src = os.path.join(test_dir, f)
    dst = f"{BASE}/test_infer/noisy/{f}"
    if os.path.isfile(src) and not os.path.exists(dst):
        os.symlink(src, dst)

print(f"Train: {len(train_files)} | Valid: {len(valid_files)}")

In [ ]:
!git clone https://github.com/sp-uhh/sgmse /kaggle/working/sgmse
%cd /kaggle/working/sgmse
!pip install -r requirements.txt

In [ ]:
!wget -O /kaggle/working/SGMSE_EN.ckpt \
  "https://huggingface.co/KhaBui/PESEM-VS/resolve/main/SGMSE_EN.skpt"
!ls -lh /kaggle/working/SGMSE_EN.ckpt

In [ ]:
# QUAN TRONG: train.py goc chi tao ModelCheckpoint khi co logger (wandb).
# Vi lenh train ben duoi dung --nolog nen logger=None -> callbacks=None -> KHONG file checkpoint nao duoc luu!
# Patch nay them 1 buoc luu checkpoint cuoi cung ngay sau trainer.fit(), bat ke co --nolog hay khong.
train_py_path = "/kaggle/working/sgmse/train.py"
s = open(train_py_path).read()

old = "     # Train model\n     trainer.fit(model, ckpt_path=args.ckpt)"
assert old in s, "Khong tim thay dong trainer.fit(...) can patch - kiem tra lai train.py cua repo."

new = old + (
    "\n\n"
    "     # --- Them: luon luu checkpoint cuoi cung sau khi finetune xong ---\n"
    "     import os as _os\n"
    "     _os.makedirs(args.log_dir, exist_ok=True)\n"
    "     _final_ckpt = _os.path.join(args.log_dir, 'finetuned_final.ckpt')\n"
    "     trainer.save_checkpoint(_final_ckpt)\n"
    "     print('Da luu checkpoint cuoi cung tai:', _final_ckpt)\n"
)

s = s.replace(old, new)
open(train_py_path, "w").write(s)
print("Da patch train.py: dam bao co checkpoint sau khi finetune du dung --nolog.")


In [ ]:
!python train.py \
  --base_dir /kaggle/working/data \
  --ckpt /kaggle/working/SGMSE_EN.ckpt \
  --backbone ncsnpp \
  --nolog \
  --batch_size 4 \
  --max_epochs 20 \
  --gpus 1

## Inference (enhancement.py tren tap test chinh thuc)

Repo SGMSE co san script `enhancement.py` de chay inference hang loat: doc tat ca wav trong `--test_dir`, chay reverse diffusion (PC sampler) va ghi enhanced wav ra `--enhanced_dir`. `test_infer/noisy` da duoc symlink tu tap TEST chinh thuc o cell dau notebook.

In [ ]:
FINAL_CKPT = "/kaggle/working/sgmse/logs/finetuned_final.ckpt"
assert os.path.isfile(FINAL_CKPT), (
    f"Khong tim thay checkpoint da finetune tai {FINAL_CKPT}. "
    "Hay chay xong cell patch + cell train (train.py) o tren truoc."
)

TEST_NOISY_DIR = f"{BASE}/test_infer/noisy"
ENHANCED_DIR = "/kaggle/working/enhanced_SGMSE"

%cd /kaggle/working/sgmse
!python enhancement.py \
  --test_dir "{TEST_NOISY_DIR}" \
  --enhanced_dir "{ENHANCED_DIR}" \
  --ckpt "{FINAL_CKPT}" \
  --device cuda


In [ ]:
print("Enhanced wav (16kHz) da duoc luu tai:", ENHANCED_DIR)
print("Buoc tiep theo: chay volume.py de RMS-normalize ve -16 dBFS, roi metrics.py de tinh PESQ/STOI/F0-RMSE/PFR.")
